In [ ]:
import pandas as pd
import warnings
import pickle

warnings.filterwarnings('ignore')

In [ ]:
with open('../data/processed/data.pkl', 'rb') as f:
    data = pickle.load(f)

target = data['target']
manual_features = data['manual_features']
standard_features = data['standard_features']
minmax_features = data['minmax_features']

## Variables

- `target` = The result aimed by model
- `manual_features` = Set of manual predict variables without scalling
- `standard_features` = Set of scalled with padronization predict variables
- `minmax_features` = Set of scalled with normalization predict variables

## Algorithms

> Separation between train and test data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
RANDOM = 5
x_train, x_test, y_train, y_test = train_test_split(minmax_features, target, test_size = 0.25, random_state=RANDOM)

#

### Naive Bayes

#### Complement Naive Bayes

In [ ]:
from sklearn.naive_bayes import ComplementNB

cnb = ComplementNB()

In [ ]:
cnb.fit(x_train, y_train)

In [ ]:
predict_cnb = cnb.predict(x_test)
accuracy_score(y_test, predict_cnb)

In [ ]:
model = cnb.predict(x_train)
accuracy_score(y_train, model)

#

#### Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

In [ ]:
gnb.fit(x_train, y_train)

In [ ]:
predict_gnb = gnb.predict(x_test)
accuracy_score(y_test, predict_gnb)

In [ ]:
model = gnb.predict(x_train)
accuracy_score(y_train, model)

#

### SVM

In [ ]:
from sklearn.svm import LinearSVC

lsvc = LinearSVC(penalty='l1',tol=0.0005, C=1.977, random_state=RANDOM)

In [ ]:
lsvc.fit(x_train, y_train)

In [ ]:
predict_lsvc = lsvc.predict(x_test)
accuracy_score(y_test, predict_lsvc)

In [ ]:
model = lsvc.predict(x_train)
accuracy_score(y_train, model)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

In [ ]:
kfold = KFold(n_splits = 50, shuffle=True, random_state = RANDOM)
model = LinearSVC(penalty='l1',tol=0.0005, C=1.977, random_state=RANDOM)

resultado = cross_val_score(model, minmax_features, target, cv = kfold)

print(f'Acuracia Media: {round((resultado.mean() * 100),2)}%')

#

## RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=40, criterion='log_loss', max_depth=3, random_state=RANDOM)

In [ ]:
rfc.fit(x_train, y_train)

In [ ]:
predict_rfc = rfc.predict(x_test)
accuracy_score(y_test, predict_rfc)

In [ ]:
model = rfc.predict(x_train)
accuracy_score(y_train, model)

### XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(max_depth=3, learning_rate=0.3, n_estimators=500, objective='binary:hinge', random_state=RANDOM, alpha=12)

In [ ]:
xgb.fit(x_train, y_train)

In [ ]:
predict_xgb = xgb.predict(x_test)
accuracy_score(y_test, predict_xgb)

In [ ]:
model = xgb.predict(x_train)
accuracy_score(y_train, model)

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty='l2',
                        tol=0.0006, C=1.95,
                        random_state=RANDOM, solver='liblinear', 
                        max_iter=200)


In [ ]:
lr.fit(x_train, y_train)

In [ ]:
predict_lr = lr.predict(x_test)
accuracy_score(y_test, predict_lr)

In [ ]:
model = lr.predict(x_train)
accuracy_score(y_train, model)

#

- LinearSVC = 98.60% Test | 98.59% Train | `minmax_features` | **LinearSVC(penalty='l1',tol=0.0005, C=1.977, random_state=RANDOM)**
- GaussianNB = 95.10% Test | 93.19% Train | `manual_features` | **GaussianNB()**
- RandomForest = 97.20% Test | 97.65% Train | `manual_features` | **RandomForestClassifier(n_estimators=40, criterion='log_loss', max_depth=3, random_state=RANDOM)**
- XGBoost = 98.60% Test | 98.12% Train | `minmax_features` | **XGBClassifier(max_depth=3, learning_rate=0.3, n_estimators=500, objective='binary:hinge', random_state=RANDOM, alpha=12)**

#

## Chosen Algorithm - SVM -> LinearSVC

#

- LinearSVC = 98.60% Test | 98.59% Train | `minmax_features` | **LinearSVC(penalty='l1',tol=0.0005, C=1.977, random_state=RANDOM)**

#

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV

params_svc = {
    'penalty': ['l1', 'l2'],
    'tol': [1e-3, 1e-4, 1e-5],
    'C': [0.1, 0.0005, 0.25, 0.002, 0.03, 0.01, 1, 10, 100, 500, 1.000, 1.977],
    'max_iter': [100, 200, 500, 1000, 2000, 2500]
}

grid_search = GridSearchCV(lsvc, params_svc, scoring='roc_auc', cv=4)
grid_search.fit(x_train, y_train)

In [ ]:
result = pd.DataFrame(grid_search.cv_results_)
result.sort_values(by='mean_test_score', ascending=False, inplace=True)
result.iloc[:, 13:]

In [ ]:
import joblib

joblib.dump(lsvc, '../model/breast-guard-model.pkl')